# Day 2-02：入力を変えて分類器を騙す

モデルを固定し、画像の画素値を更新します。

## 実行前の準備

**`DRIVE_PROJECT`を自分のDrive内の教材フォルダに合わせてください。**
ローカルでは`notebooks/day2/`または教材ルートから実行します。ColabではGPUランタイムを選びます。

In [ ]:
#@title 環境設定（最初に実行）
from pathlib import Path
import runpy

###################################################################
# JKJ1A/へ移動．
DRIVE_PROJECT = Path("/content/drive/MyDrive/Colab Notebooks/JKJ1A")  # !!! 自分の環境に合わせて変更
LOCAL_PROJECT = (
    Path.cwd()
    if (Path.cwd() / "jkj1a/launcher.py").is_file()
    else Path.cwd().parent.parent
)
###################################################################

try:
    from google.colab import drive
    ON_COLAB = True
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    project_hint = DRIVE_PROJECT
except ImportError:
    ON_COLAB = False
    project_hint = LOCAL_PROJECT

_context = runpy.run_path(str(project_hint / "jkj1a/launcher.py"))
ON_COLAB = _context["ON_COLAB"]
project_root = _context["project_root"]
print("実行場所:", "Colab" if ON_COLAB else "ローカル")
print("作業フォルダ:", project_root)

## 1. 学習済みモデルと画像

01で保存した重みを読み込みます。まず変更前に正しく分類できているか確認します。

In [ ]:
from jkj1a.classification import load_classifier
from jkj1a.classification import load_cifar10
import torch
from torch.nn import functional as F

# 01で学習したモデルを読み込む
model = load_classifier()
device = next(model.parameters()).device

# 攻撃する画像を取り出す
_, test_loader = load_cifar10(batch_size=32)
x, y = next(iter(test_loader))
x, y = x.to(device), y.to(device)

# 勾配を計算せず、変更前に正しく分類できる画像を調べる
with torch.no_grad():
    correct = model(x).argmax(1) == y
print("変更前の正解数:", correct.sum().item(), "/", len(y))
if not correct.any():
    raise ValueError("このバッチでは全問不正解です。別のバッチを選ぶか、01で学習を続けてください。")

# 変更前に正解している画像を表示用に選ぶ
index = correct.nonzero(as_tuple=True)[0][0].item()

## 2. FGSM：画像の勾配を使う

学習では重みを変えて損失を小さくしました。ここでは画像$x$を変えて、正解ラベル$y$に対する損失を大きくします。

$$x_{\mathrm{adv}}=\operatorname{clip}_{[0,1]}\left(x+\epsilon\,\operatorname{sign}(\nabla_x L(f(x),y))\right)$$

`eps=8/255`は、0〜1で表した各画素を最大8/255だけ変える制約です。正規化はモデルの内部で行います。

### 演習：FGSMを実装する

上の式に従って、`fgsm`関数を完成させてください。正解ラベルに対する交差エントロピー損失を使い、画像に関する勾配を求めます。各画素を勾配の符号に沿って`eps`だけ動かし、画素値を0〜1に収めて返してください。元画像とモデルの重みは変更しません。


In [ ]:
def fgsm(model, x, y, *, eps=8/255):
    model.eval()

    # 元画像をコピーし、画像に関する勾配を計算できるようにする
    x_adv = x.detach().clone().requires_grad_(True)

    # 損失と、画像に関する勾配を計算する
    

    # 勾配を使って画像を更新し、画素値の範囲を制限する
    

    # 画素値の範囲[0, 1]に制限する
    x_adv = x_adv.clamp(0, 1)  
    
    return x_adv


In [ ]:
from jkj1a.classification import CIFAR_CLASSES
from jkj1a.adversarial import show_attack

# FGSMで、正解クラスから外れるように画像を1回更新する
x_adv = fgsm(model, x, y, eps=8/255)

# 選んだ画像について、元画像・強調した摂動・攻撃画像と予測結果を表示する
show_attack(model, x, x_adv, CIFAR_CLASSES, index=index)
print("画素の最大変化:", (x_adv - x).abs().max().item())


### 演習：Targeted攻撃を実装しよう

上記の`fgsm`関数は，正解から遠ざける（誤差`loss`を悪化させる）ことで間違いを引き起こします．ただし，これでは間違った結果どのクラスに分類されるかまではわかりません．「Targetedな攻撃」とは誤り先を指定する方法です．これは，誤り先クラスでの誤差を改善することで実現できます（結果として元のクラスの誤差は悪化する）．

**備考：**
- 上記の`fgsm`関数に対して非常に短い修正を行うことで実現できます．`targeted`というキーワード引数と`if`文を使います．
- ラベル`y`の意味も変わります
- 参考：[敵対的サンプル入門](https://nonstop-umbrella-d21.notion.site/3bcf6af2a17c41138aa4aa215a918c53)

In [ ]:
def fgsm(model, x, y, *, eps=8/255, targeted=False):
    pass

全ての画像をネコに誤分類させてください

In [ ]:
# 画像の準備
_, test_loader = load_cifar10(batch_size=32)
x, y = next(iter(test_loader))
x, y = x.to(device), y.to(device)



## 3. I-FGSM：小さな更新を繰り返す


Iterative FGSM (I-FGSM)はFSGMを繰り返し行い，さらに強力な摂動を生成します．

- `eps` - 元画像から許される変化の上限
- `alpha` - 1回あたり許される更新幅
- `n_setps` - 更新回数

標準的には，`eps=8/255`, `alpha = eps/4`, `n_steps = 10`程度に設定することが多いです（CIFAR-10画像に対しては）．

---

**演習1：**

I-FGSMを実装して下さい

備考
- 内部で上記の`fgsm`関数を呼ばない方がいいです（元画像からのズレが制御しづらい）

In [ ]:
def ifgsm(model, x, y, *, eps=8/255, alpha=2/255, n_steps=10, targeted=False):
    """L-infinity I-FGSM; x is in [0,1], y is the target for targeted attacks."""

    # 引数のチェック
    if eps < 0 or alpha < 0 or not isinstance(n_steps, int) or n_steps < 1:
        raise ValueError('eps and alpha must be nonnegative; n_steps must be a positive integer')

    was_training = model.training
    model.eval()

    # 許容範囲の基準となる元画像を残す
    original = x.detach()
    adv = original.clone()

    try:

        for _ in range(n_steps):

            # FGSMとほぼ同じ実装
            pass

    finally:

        # 呼び出し前のモードに戻す
        model.train(was_training)

    return adv

In [ ]:
# 小さな更新を10回繰り返して攻撃画像を作る
x_adv = ifgsm(model, x, y, eps=8/255, alpha=2/255, n_steps=10)

# 選んだ画像について、元画像・強調した摂動・攻撃画像と予測結果を表示する
show_attack(model, x, x_adv, CIFAR_CLASSES, index=index)
print("画素の最大変化:", (x_adv-x).abs().max().item())

## 4. 複数の画像で評価する

同じテスト画像を使って比較します。`adversarial_accuracy`は攻撃後の正解率です。`success_rate`は**変更前に正解していた画像のうち、攻撃後に不正解となった割合**です。初めから間違えていた画像は攻撃成功に数えません。

ここでは先頭8バッチを評価します。全体を調べる場合は`max_batches=None`にします。

In [ ]:
from jkj1a.adversarial import evaluate_attack

results = []

# 画素を変えてよい幅を変え、攻撃の強さによる違いを調べる
for eps in (0, 1/255, 2/255, 4/255, 8/255, 16/255):
    # 先頭8バッチで攻撃前後の正解率と、元は正解した画像への攻撃成功率を求める
    row = evaluate_attack(model, test_loader, attack=ifgsm,
                          eps=eps, alpha=eps/4, n_steps=10, max_batches=8)
    results.append({"eps": eps, **row})
    print(results[-1], flush=True)

## 演習

`test_loader`に含まれる画像全体に渡りI-FGSMを行い，分類精度がどのように低下するか様子を観察したい

1. 攻撃強度`eps`を横軸，分類精度を縦軸にしたプロットを作り，分類精度の低下の様子を確認せよ
2. 敵対的摂動など使わずとも，「適当な摂動」で分類精度は十分低下するのではないか？　I-FGSMの代わりに，単純に`[-eps, eps]`の一様乱数で摂動を加えた画像を作り、上のプロットに重ねて比較せよ
3. 全ての画像をネコとして誤分類させたい．ネコと誤分類させやすいクラスとそうでないものはあるだろうか？ TargetedなI-FGSMを使って検証せよ

In [ ]:
# ここにコードを書く